<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.3/blob/main/05_Final_exploration_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 05_Final_exploration_analysis.py
# ============================================================
#
# FINAL LEAKAGE-SAFE MATERIALS EXPLORATION
#
# ============================================================
#
# DESIGN
# ------
#
# 43 experimentally measured copolymers
#
# Each repeat:
#     Initial set = 30
#     Holdout set = 13
#
# Repeats:
#     200
#
# Selection:
#     Sequential maximin
#
#
# FOUR CONDITIONS
# ---------------
#
# N0:
#     RDKit Numeric
#     + TD-NMR Numeric
#
# N1:
#     RDKit Numeric
#     + TD-NMR Numeric
#     + Chemical Language
#
# N2:
#     RDKit Numeric
#     + TD-NMR Numeric
#     + TD Dynamic-State Language
#
# N3:
#     RDKit Numeric
#     + TD-NMR Numeric
#     + Chemical Language
#     + TD Dynamic-State Language
#
#
# INDEPENDENT Y
# -------------
#
# Solution-NMR is NEVER used for:
#
#     representation construction
#     scaling
#     language generation
#     maximin selection
#
# It is used ONLY after selection for evaluation.
#
#
# FIXED Y SPACES
# --------------
#
# Primary:
#
#     Glycerol × Alkyl-chain
#
# Secondary:
#
#     Glycerol × Alkenyl
#
#
# MAIN OUTPUTS
# ------------
#
# Coverage trajectory
# Coverage AUC
# k = 3 / 5 / 7 / 9 coverage
#
# Composition-ratio variability
# Role-identity diversity
# Pairwise formulation diversity
#
# Early Top-5 selection probability
#
# Paired Wilcoxon
# BH-FDR
# Rank-biserial effect size
# Win rate
#
# Factorial language effects
#
# ============================================================


# ============================================================
# 0. INSTALL / IMPORT
# ============================================================

import sys
import subprocess
import importlib.util


def ensure_package(import_name, pip_name=None):

    if pip_name is None:
        pip_name = import_name

    if importlib.util.find_spec(import_name) is None:

        print(f"[INSTALL] {pip_name}")

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                pip_name,
            ]
        )


ensure_package("numpy")
ensure_package("pandas")
ensure_package("scipy")
ensure_package("sklearn", "scikit-learn")
ensure_package("sentence_transformers", "sentence-transformers")
ensure_package("openpyxl")
ensure_package("matplotlib")


import os
import re
import json
import shutil
import zipfile
import warnings

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import wilcoxon

from sklearn.preprocessing import StandardScaler

from sentence_transformers import SentenceTransformer


warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

EXPECTED_N = 43

N_REPEATS = 200

INITIAL_N = 30

HOLDOUT_N = 13

RANDOM_STATE = 42

EARLY_K = 5

LANDMARK_K = [
    3,
    5,
    7,
    9,
]

LANGUAGE_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)


# ============================================================
# 2. OUTPUT
# ============================================================

OUTPUT_DIR = Path(
    "05_Final_exploration_analysis_output"
)

ZIP_PATH = Path(
    "05_Final_exploration_analysis_output.zip"
)


if OUTPUT_DIR.exists():

    shutil.rmtree(
        OUTPUT_DIR
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


if ZIP_PATH.exists():

    ZIP_PATH.unlink()


# ============================================================
# 3. CONDITIONS
# ============================================================

CONDITIONS = [

    "N0_Numeric",

    "N1_ChemicalLanguage",

    "N2_DynamicStateLanguage",

    "N3_BothLanguages",
]


CONDITION_LABELS = {

    "N0_Numeric":
        "N0: Numeric",

    "N1_ChemicalLanguage":
        "N1: + Chemical Language",

    "N2_DynamicStateLanguage":
        "N2: + Dynamic-State Language",

    "N3_BothLanguages":
        "N3: + Both Languages",
}


# ============================================================
# 4. FIXED Y SPACES
# ============================================================

Y_SPACES = {

    "Glycerol_x_AlkylChain": [

        "Glycerol_Response",

        "AlkylChain_Response",
    ],

    "Glycerol_x_Alkenyl": [

        "Glycerol_Response",

        "Alkenyl_Response",
    ],
}


# ============================================================
# 5. TD-NMR NUMERIC DESCRIPTORS
# ============================================================

TD_NUMERIC_COLUMNS = [

    "ShortFraction",

    "MidFraction",

    "LongFraction",

    "Weighted_logmean_T2_ms",

    "Width_log10T2",

    "N_detected_peaks",

    "Entropy_norm",
]


# ============================================================
# 6. RDKit NUMERIC DESCRIPTORS
# ============================================================

RDKIT_COLUMNS = [

    "cLogP",

    "TPSA",

    "Hbond",

    "RotatableBonds",

    "IonicCharacter",

    "HeteroatomFraction",
]


# ============================================================
# 7. HELPERS
# ============================================================

def canonical_id(x):

    s = str(x).strip()

    s = s.replace(
        "-",
        "_"
    )

    s = re.sub(
        r"\s+",
        "",
        s
    )

    return s.upper()


def robust_read_csv(path):

    last_error = None

    for encoding in [

        "utf-8-sig",

        "utf-8",

        "cp932",

        "latin1",
    ]:

        try:

            return pd.read_csv(
                path,
                encoding=encoding
            )

        except Exception as e:

            last_error = e

    raise last_error


def find_file(
    paths,
    keyword
):

    matches = [

        p
        for p in paths

        if keyword.lower()
        in p.name.lower()
    ]

    if len(matches) == 0:

        raise FileNotFoundError(
            f"Could not find file containing: "
            f"{keyword}"
        )

    return matches[0]


# ============================================================
# 8. UPLOAD INPUTS
# ============================================================
#
# REQUIRED INPUTS
#
# From 01:
#   01_TD_NMR_descriptors.csv
#
# From 02:
#   01_Dynamic_State_Language.csv
#   (used for audit / definition only)
#
# From 03:
#   Solution-NMR final Y CSV
#
# From 04:
#   03_RDKit_Numeric_Final.csv
#   04_Chemical_Language_Sentences.csv
#   05_Chemical_Language_Embedding_384D.csv
#
# Original:
#   copolymer_composition.csv
#
# ============================================================

try:

    from google.colab import files

    print("=" * 80)

    print(
        "Upload the following files:"
    )

    print(
        "1) 01_TD_NMR_descriptors.csv"
    )

    print(
        "2) 01_Dynamic_State_Language.csv"
    )

    print(
        "3) Final Solution-NMR Y CSV"
    )

    print(
        "4) 03_RDKit_Numeric_Final.csv"
    )

    print(
        "5) 04_Chemical_Language_Sentences.csv"
    )

    print(
        "6) 05_Chemical_Language_Embedding_384D.csv"
    )

    print(
        "7) copolymer_composition.csv"
    )

    print("=" * 80)

    uploaded = files.upload()

    uploaded_paths = [

        Path(x)

        for x in uploaded.keys()
    ]

except ImportError:

    uploaded_paths = list(
        Path(".").glob("*.csv")
    )


# ============================================================
# 9. IDENTIFY FILES
# ============================================================

TD_FILE = find_file(

    uploaded_paths,

    "TD_NMR_descriptors"
)


TD_LANGUAGE_AUDIT_FILE = find_file(

    uploaded_paths,

    "Dynamic_State_Language"
)


RDKIT_FILE = find_file(

    uploaded_paths,

    "RDKit_Numeric_Final"
)


CHEM_SENTENCE_FILE = find_file(

    uploaded_paths,

    "Chemical_Language_Sentences"
)


CHEM_EMBED_FILE = find_file(

    uploaded_paths,

    "Chemical_Language_Embedding"
)


COMPOSITION_FILE = find_file(

    uploaded_paths,

    "copolymer_composition"
)


# ------------------------------------------------------------
# Solution-NMR Y
# ------------------------------------------------------------

y_candidates = [

    p
    for p in uploaded_paths

    if (
        "solutionnmr"
        in p.name.lower()

        and

        (
            "endpoint"
            in p.name.lower()

            or

            "independent_y"
            in p.name.lower()

            or

            "code10"
            in p.name.lower()
        )
    )
]


if len(y_candidates) == 0:

    raise FileNotFoundError(
        "Could not identify final Solution-NMR Y CSV."
    )


Y_FILE = y_candidates[0]


print("\nDetected inputs")
print("----------------")

for label, path in [

    ("TD numeric", TD_FILE),

    ("TD language audit", TD_LANGUAGE_AUDIT_FILE),

    ("Solution-NMR Y", Y_FILE),

    ("RDKit", RDKIT_FILE),

    ("Chemical sentences", CHEM_SENTENCE_FILE),

    ("Chemical embedding", CHEM_EMBED_FILE),

    ("Composition", COMPOSITION_FILE),
]:

    print(
        f"{label:24s}: {path.name}"
    )


# ============================================================
# 10. LOAD INPUTS
# ============================================================

td = robust_read_csv(
    TD_FILE
)

td_lang_audit = robust_read_csv(
    TD_LANGUAGE_AUDIT_FILE
)

y_df = robust_read_csv(
    Y_FILE
)

rdkit_df = robust_read_csv(
    RDKIT_FILE
)

chem_sentence_df = robust_read_csv(
    CHEM_SENTENCE_FILE
)

chem_embedding_df = robust_read_csv(
    CHEM_EMBED_FILE
)

composition_df = robust_read_csv(
    COMPOSITION_FILE
)


# ============================================================
# 11. CANONICAL-ID STANDARDIZATION
# ============================================================

def ensure_canonical_id(
    df
):

    candidates = [

        "Canonical_ID",

        "Copolymer_Name",

        "Sample_Key",

        "Sample_Name",
    ]

    source = next(

        (
            c
            for c in candidates
            if c in df.columns
        ),

        None
    )

    if source is None:

        raise ValueError(
            "No material identifier column."
        )

    df = df.copy()

    df[
        "Canonical_ID"
    ] = (

        df[
            source
        ]

        .map(
            canonical_id
        )
    )

    return df


td = ensure_canonical_id(
    td
)

td_lang_audit = ensure_canonical_id(
    td_lang_audit
)

y_df = ensure_canonical_id(
    y_df
)

rdkit_df = ensure_canonical_id(
    rdkit_df
)

chem_sentence_df = ensure_canonical_id(
    chem_sentence_df
)

chem_embedding_df = ensure_canonical_id(
    chem_embedding_df
)

composition_df = ensure_canonical_id(
    composition_df
)


# ============================================================
# 12. SOLUTION-NMR Y COLUMN NORMALIZATION
# ============================================================
#
# Accept either:
#
# final WithinRegime aliases:
#   Glycerol_Response
#   AlkylChain_Response
#   Alkenyl_Response
#
# OR original names:
#
# glycerol_oxygenated_ch_WithinRegime_Z
# alkyl_chain_chx_WithinRegime_Z
# alkenyl_ch_WithinRegime_Z
#
# ============================================================

Y_ALIASES = {

    "Glycerol_Response": [

        "Glycerol_Response",

        "glycerol_oxygenated_ch_WithinRegime_Z",
    ],

    "AlkylChain_Response": [

        "AlkylChain_Response",

        "alkyl_chain_chx_WithinRegime_Z",
    ],

    "Alkenyl_Response": [

        "Alkenyl_Response",

        "alkenyl_ch_WithinRegime_Z",
    ],
}


for target, candidates in (
    Y_ALIASES.items()
):

    source = next(

        (
            c
            for c in candidates

            if c in y_df.columns
        ),

        None
    )

    if source is None:

        raise ValueError(

            f"Required final Solution-NMR "
            f"endpoint not found: {target}"
        )

    y_df[
        target
    ] = pd.to_numeric(

        y_df[
            source
        ],

        errors="coerce"
    )


# ============================================================
# 13. CHEMICAL EMBEDDING COLUMNS
# ============================================================

chem_embedding_cols = [

    c

    for c in chem_embedding_df.columns

    if c.startswith(
        "ChemLang_"
    )
]


if len(
    chem_embedding_cols
) == 0:

    # fallback:
    # all numeric columns except IDs

    chem_embedding_cols = [

        c

        for c in chem_embedding_df.columns

        if (
            c
            not in [
                "Canonical_ID",
                "Copolymer_Name",
            ]

            and

            pd.api.types.is_numeric_dtype(
                chem_embedding_df[c]
            )
        )
    ]


if len(
    chem_embedding_cols
) == 0:

    raise ValueError(
        "Chemical Language embedding columns not found."
    )


print(
    "\nChemical Language dimensions:",
    len(
        chem_embedding_cols
    )
)


# ============================================================
# 14. COMPOSITION COLUMNS
# ============================================================

required_comp = [

    "mono_1",

    "mono_2",

    "mono_3",

    "comp_1",

    "comp_2",

    "comp_3",
]


missing_comp = [

    c
    for c in required_comp

    if c not in composition_df.columns
]


if missing_comp:

    raise ValueError(
        "Composition file missing:\n"
        +
        "\n".join(
            missing_comp
        )
    )


for c in [

    "comp_1",

    "comp_2",

    "comp_3",
]:

    composition_df[
        c
    ] = pd.to_numeric(

        composition_df[
            c
        ],

        errors="coerce"
    )


# ============================================================
# 15. MERGE MASTER TABLE
# ============================================================

master = (

    td[
        [
            "Canonical_ID",
            *TD_NUMERIC_COLUMNS,
        ]
    ]

    .merge(

        rdkit_df[
            [
                "Canonical_ID",
                *RDKIT_COLUMNS,
            ]
        ],

        on="Canonical_ID",

        how="inner"
    )

    .merge(

        y_df[
            [
                "Canonical_ID",
                "Glycerol_Response",
                "AlkylChain_Response",
                "Alkenyl_Response",
            ]
        ],

        on="Canonical_ID",

        how="inner"
    )

    .merge(

        composition_df[
            [
                "Canonical_ID",
                *required_comp,
            ]
        ],

        on="Canonical_ID",

        how="inner"
    )
)


master = master.merge(

    chem_embedding_df[
        [
            "Canonical_ID",
            *chem_embedding_cols,
        ]
    ],

    on="Canonical_ID",

    how="inner"
)


# ============================================================
# 16. STRICT MASTER QC
# ============================================================

if (
    master[
        "Canonical_ID"
    ]
    .duplicated()
    .any()
):

    raise ValueError(
        "Duplicate Canonical_ID after merge."
    )


print(
    "\nMerged materials:",
    len(
        master
    )
)


if len(
    master
) != EXPECTED_N:

    all_sets = {

        "TD":
            set(
                td[
                    "Canonical_ID"
                ]
            ),

        "RDKit":
            set(
                rdkit_df[
                    "Canonical_ID"
                ]
            ),

        "Y":
            set(
                y_df[
                    "Canonical_ID"
                ]
            ),

        "ChemicalLanguage":
            set(
                chem_embedding_df[
                    "Canonical_ID"
                ]
            ),

        "Composition":
            set(
                composition_df[
                    "Canonical_ID"
                ]
            ),
    }


    union = set().union(
        *all_sets.values()
    )


    print(
        "\nMaterial merge audit:"
    )


    for name, ids in (
        all_sets.items()
    ):

        missing_ids = sorted(
            union
            -
            ids
        )

        print(
            f"\n{name} missing:"
        )

        print(
            missing_ids
        )


    raise ValueError(

        f"Expected {EXPECTED_N} merged materials "
        f"but obtained {len(master)}."
    )


# ============================================================
# 17. NUMERIC QC
# ============================================================

numeric_required = (

    TD_NUMERIC_COLUMNS

    +

    RDKIT_COLUMNS

    +

    [
        "Glycerol_Response",
        "AlkylChain_Response",
        "Alkenyl_Response",
        "comp_1",
        "comp_2",
        "comp_3",
    ]

    +

    chem_embedding_cols
)


if (
    master[
        numeric_required
    ]
    .isna()
    .any()
    .any()
):

    bad_columns = (

        master[
            numeric_required
        ]
        .isna()
        .sum()
    )

    bad_columns = (
        bad_columns[
            bad_columns > 0
        ]
    )

    raise ValueError(

        "Missing values detected in final analysis table:\n"

        + bad_columns.to_string()
    )


# ============================================================
# 18. SORT MATERIALS
# ============================================================

master = (

    master

    .sort_values(
        "Canonical_ID"
    )

    .reset_index(
        drop=True
    )
)


ids = master[
    "Canonical_ID"
].tolist()


N = len(
    master
)


# ============================================================
# 19. LOAD SENTENCE MODEL
# ============================================================

print(
    "\nLoading SentenceTransformer..."
)


language_model = SentenceTransformer(
    LANGUAGE_MODEL_NAME
)


# ============================================================
# 20. DISTANCE HELPERS
# ============================================================

def euclidean_distance_matrix(
    X
):

    X = np.asarray(
        X,
        dtype=float
    )

    sq = np.sum(
        X ** 2,
        axis=1,
        keepdims=True
    )

    D2 = (
        sq
        +
        sq.T
        -
        2.0
        *
        X.dot(
            X.T
        )
    )

    D2 = np.maximum(
        D2,
        0.0
    )

    return np.sqrt(
        D2
    )


def median_training_distance(
    D,
    train_idx
):

    sub = D[
        np.ix_(
            train_idx,
            train_idx
        )
    ]

    vals = sub[
        np.triu_indices(
            len(
                train_idx
            ),
            k=1
        )
    ]

    vals = vals[
        np.isfinite(
            vals
        )
    ]

    vals = vals[
        vals > 0
    ]

    if len(
        vals
    ) == 0:

        return 1.0

    med = float(
        np.median(
            vals
        )
    )

    if (
        not np.isfinite(
            med
        )

        or

        med <= 1e-12
    ):

        return 1.0

    return med


def normalize_distance_block(
    D,
    train_idx
):

    med = median_training_distance(
        D,
        train_idx
    )

    return (
        D / med,
        med
    )


def combine_distance_blocks(
    blocks
):

    if len(
        blocks
    ) == 0:

        raise ValueError(
            "No distance blocks."
        )

    stack = np.stack(
        blocks,
        axis=0
    )

    # RMS block balancing
    #
    # Each block contributes equally irrespective
    # of dimensionality.

    return np.sqrt(
        np.mean(
            stack ** 2,
            axis=0
        )
    )


# ============================================================
# 21. TRAINING-ONLY NUMERIC BLOCK
# ============================================================

def numeric_block_distance(
    X,
    train_idx
):

    X = np.asarray(
        X,
        dtype=float
    )

    scaler = StandardScaler()

    scaler.fit(
        X[
            train_idx
        ]
    )

    Z = scaler.transform(
        X
    )

    D = euclidean_distance_matrix(
        Z
    )

    D_norm, median_distance = (
        normalize_distance_block(
            D,
            train_idx
        )
    )

    return (
        D_norm,
        median_distance
    )


# ============================================================
# 22. CHEMICAL LANGUAGE DISTANCE
# ============================================================
#
# Chemical Language is Y-independent and fixed.
#
# Embeddings were normalized by 04.
#
# Only the distance-scale normalization uses initial30.
#
# ============================================================

X_chem_lang = (

    master[
        chem_embedding_cols
    ]
    .to_numpy(
        dtype=float
    )
)


D_chem_lang_raw = (
    euclidean_distance_matrix(
        X_chem_lang
    )
)


# ============================================================
# 23. TD DYNAMIC-STATE LANGUAGE
# ============================================================
#
# IMPORTANT:
#
# Thresholds are fitted ONLY using the initial 30
# of each repeat.
#
# Weighted_logmean_T2_ms is deliberately NOT directly
# verbalized.
#
# ============================================================

def quantile_safe(
    x,
    q
):

    x = np.asarray(
        x,
        dtype=float
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    if len(
        x
    ) == 0:

        return np.nan

    return float(
        np.quantile(
            x,
            q
        )
    )


def fit_td_language_rules(
    td_dataframe,
    train_idx
):

    train = td_dataframe.iloc[
        train_idx
    ]


    rules = {

        "Width_q33":
            quantile_safe(
                train[
                    "Width_log10T2"
                ],
                1 / 3
            ),

        "Width_q67":
            quantile_safe(
                train[
                    "Width_log10T2"
                ],
                2 / 3
            ),

        "Peaks_q33":
            quantile_safe(
                train[
                    "N_detected_peaks"
                ],
                1 / 3
            ),

        "Peaks_q67":
            quantile_safe(
                train[
                    "N_detected_peaks"
                ],
                2 / 3
            ),

        "Entropy_q33":
            quantile_safe(
                train[
                    "Entropy_norm"
                ],
                1 / 3
            ),

        "Entropy_q67":
            quantile_safe(
                train[
                    "Entropy_norm"
                ],
                2 / 3
            ),
    }

    return rules


def three_state(
    value,
    q33,
    q67,
    low_label,
    mid_label,
    high_label
):

    if value <= q33:

        return low_label

    if value <= q67:

        return mid_label

    return high_label


def population_balance_sentence(
    short_fraction,
    mid_fraction,
    long_fraction
):

    fractions = {

        "short-T2":
            float(
                short_fraction
            ),

        "intermediate-T2":
            float(
                mid_fraction
            ),

        "long-T2":
            float(
                long_fraction
            ),
    }


    ordered = sorted(

        fractions.items(),

        key=lambda x:
            x[1],

        reverse=True
    )


    first_name, first_value = (
        ordered[0]
    )

    second_name, second_value = (
        ordered[1]
    )


    if first_value >= 0.60:

        return (

            f"The relaxation population is "
            f"dominated by the {first_name} component."
        )


    if abs(
        first_value
        -
        second_value
    ) <= 0.10:

        return (

            f"The relaxation population shows "
            f"coexisting {first_name} and "
            f"{second_name} components."
        )


    return (

        f"The relaxation population is distributed "
        f"across several dynamic components, with "
        f"the {first_name} component being the largest."
    )


def topology_sentence(
    width,
    peaks,
    rules
):

    width_state = three_state(

        width,

        rules[
            "Width_q33"
        ],

        rules[
            "Width_q67"
        ],

        "narrow",

        "intermediate-width",

        "broad"
    )


    peak_state = three_state(

        peaks,

        rules[
            "Peaks_q33"
        ],

        rules[
            "Peaks_q67"
        ],

        "few-feature",

        "moderately structured",

        "multi-feature"
    )


    return (

        f"The relaxation distribution has a "
        f"{width_state}, {peak_state} topology."
    )


def coexistence_sentence(
    short_fraction,
    mid_fraction,
    long_fraction
):

    fractions = np.array(

        [
            short_fraction,
            mid_fraction,
            long_fraction,
        ],

        dtype=float
    )


    active = int(
        np.sum(
            fractions >= 0.15
        )
    )


    if active >= 3:

        return (

            "Short-, intermediate-, and long-T2 "
            "dynamic states coexist substantially."
        )


    if active == 2:

        return (

            "Two major dynamic-state populations "
            "coexist in the relaxation profile."
        )


    return (

        "The relaxation profile is dominated by "
        "a single major dynamic-state population."
    )


def heterogeneity_sentence(
    width,
    entropy,
    rules
):

    width_state = three_state(

        width,

        rules[
            "Width_q33"
        ],

        rules[
            "Width_q67"
        ],

        "low-width",

        "intermediate-width",

        "high-width"
    )


    entropy_state = three_state(

        entropy,

        rules[
            "Entropy_q33"
        ],

        rules[
            "Entropy_q67"
        ],

        "low-entropy",

        "intermediate-entropy",

        "high-entropy"
    )


    if (
        width_state == "high-width"

        and

        entropy_state == "high-entropy"
    ):

        level = (
            "high dynamic heterogeneity"
        )


    elif (
        width_state == "low-width"

        and

        entropy_state == "low-entropy"
    ):

        level = (
            "low dynamic heterogeneity"
        )


    else:

        level = (
            "intermediate dynamic heterogeneity"
        )


    return (

        f"The combined distribution width and "
        f"entropy indicate {level}."
    )


def build_td_language(
    td_dataframe,
    rules
):

    sentences = []


    for _, row in (
        td_dataframe.iterrows()
    ):

        pieces = [

            population_balance_sentence(

                row[
                    "ShortFraction"
                ],

                row[
                    "MidFraction"
                ],

                row[
                    "LongFraction"
                ],
            ),

            topology_sentence(

                row[
                    "Width_log10T2"
                ],

                row[
                    "N_detected_peaks"
                ],

                rules,
            ),

            coexistence_sentence(

                row[
                    "ShortFraction"
                ],

                row[
                    "MidFraction"
                ],

                row[
                    "LongFraction"
                ],
            ),

            heterogeneity_sentence(

                row[
                    "Width_log10T2"
                ],

                row[
                    "Entropy_norm"
                ],

                rules,
            ),
        ]


        sentences.append(
            " ".join(
                pieces
            )
        )


    return sentences


# ============================================================
# 24. MAXIMIN
# ============================================================

def sequential_maximin(
    D,
    initial_idx,
    holdout_idx
):

    selected = list(
        initial_idx
    )

    remaining = list(
        holdout_idx
    )

    order = []


    while len(
        remaining
    ) > 0:

        scores = []


        for candidate in (
            remaining
        ):

            dmin = float(

                np.min(

                    D[
                        candidate,
                        selected
                    ]
                )
            )


            scores.append(
                (
                    dmin,
                    candidate
                )
            )


        # deterministic tie-breaking:
        #
        # highest distance first,
        # then smallest material index

        scores.sort(

            key=lambda x:
                (
                    -x[0],
                    x[1],
                )
        )


        chosen = scores[
            0
        ][1]


        order.append(
            chosen
        )

        selected.append(
            chosen
        )

        remaining.remove(
            chosen
        )


    return order


# ============================================================
# 25. COVERAGE METRIC
# ============================================================
#
# Coverage:
#
# 1 - RMS nearest-selected distance /
#     full-space diameter
#
# Y-space is standardized using all 43 Y values
# because Y is used only for post-selection evaluation,
# never for representation construction or selection.
#
# ============================================================

def prepare_y_space(
    df,
    columns
):

    X = (

        df[
            columns
        ]
        .to_numpy(
            dtype=float
        )
    )


    mean = np.mean(
        X,
        axis=0
    )

    sd = np.std(
        X,
        axis=0,
        ddof=0
    )

    sd[
        sd <= 1e-12
    ] = 1.0


    Z = (
        X
        -
        mean
    ) / sd


    D = euclidean_distance_matrix(
        Z
    )


    diameter = float(
        np.max(
            D
        )
    )


    if diameter <= 1e-12:

        diameter = 1.0


    return (
        Z,
        D,
        diameter
    )


def coverage_score(
    D_y,
    selected_idx,
    diameter
):

    selected_idx = list(
        selected_idx
    )


    nearest = np.min(

        D_y[
            :,
            selected_idx
        ],

        axis=1
    )


    rms = float(

        np.sqrt(

            np.mean(
                nearest ** 2
            )
        )
    )


    score = (
        1.0
        -
        rms
        /
        diameter
    )


    return float(
        np.clip(
            score,
            0.0,
            1.0
        )
    )


def coverage_trajectory(
    D_y,
    initial_idx,
    selection_order,
    diameter
):

    trajectory = []


    selected = list(
        initial_idx
    )


    # k = 0
    trajectory.append(

        coverage_score(

            D_y,

            selected,

            diameter
        )
    )


    for idx in (
        selection_order
    ):

        selected.append(
            idx
        )


        trajectory.append(

            coverage_score(

                D_y,

                selected,

                diameter
            )
        )


    return np.asarray(
        trajectory,
        dtype=float
    )


def normalized_auc(
    trajectory
):

    x = np.arange(
        len(
            trajectory
        ),
        dtype=float
    )


    if x[-1] == 0:

        return float(
            trajectory[0]
        )


    return float(

        np.trapz(
            trajectory,
            x
        )

        /

        x[-1]
    )


# ============================================================
# 26. COMPOSITION DIVERSITY
# ============================================================

def shannon_entropy_from_counts(
    values
):

    _, counts = np.unique(

        np.asarray(
            values,
            dtype=str
        ),

        return_counts=True
    )


    p = (
        counts
        /
        counts.sum()
    )


    H = float(

        -np.sum(

            p
            *
            np.log(
                p
            )
        )
    )


    if len(
        counts
    ) <= 1:

        return 0.0


    return (

        H
        /
        np.log(
            len(
                counts
            )
        )
    )


def role_identity_diversity(
    selected_df
):

    values = []


    for role in [

        "mono_1",

        "mono_2",

        "mono_3",
    ]:

        values.append(

            shannon_entropy_from_counts(

                selected_df[
                    role
                ]
            )
        )


    return float(
        np.mean(
            values
        )
    )


def composition_ratio_variability(
    selected_df
):

    X = (

        selected_df[
            [
                "comp_1",
                "comp_2",
                "comp_3",
            ]
        ]

        .to_numpy(
            dtype=float
        )
    )


    if len(
        X
    ) <= 1:

        return 0.0


    # Mean SD across the three composition roles

    return float(

        np.mean(

            np.std(
                X,
                axis=0,
                ddof=1
            )
        )
    )


def pairwise_formulation_diversity(
    selected_df
):

    X = (

        selected_df[
            [
                "comp_1",
                "comp_2",
                "comp_3",
            ]
        ]

        .to_numpy(
            dtype=float
        )
    )


    if len(
        X
    ) <= 1:

        return 0.0


    D = euclidean_distance_matrix(
        X
    )


    vals = D[
        np.triu_indices(
            len(
                X
            ),
            k=1
        )
    ]


    return float(
        np.mean(
            vals
        )
    )


def composite_design_diversity(
    selected_df
):

    identity = (
        role_identity_diversity(
            selected_df
        )
    )


    ratio = (
        composition_ratio_variability(
            selected_df
        )
    )


    return (
        identity,
        ratio
    )


# ============================================================
# 27. STATISTICAL HELPERS
# ============================================================

def rank_biserial_paired(
    differences
):

    d = np.asarray(
        differences,
        dtype=float
    )

    d = d[
        np.isfinite(
            d
        )
    ]


    d = d[
        d != 0
    ]


    if len(
        d
    ) == 0:

        return 0.0


    abs_d = np.abs(
        d
    )


    order = np.argsort(
        abs_d
    )


    ranks = np.empty(
        len(
            d
        ),
        dtype=float
    )


    # simple average-rank implementation

    sorted_abs = abs_d[
        order
    ]


    i = 0

    while i < len(
        d
    ):

        j = i + 1

        while (
            j < len(
                d
            )

            and

            np.isclose(
                sorted_abs[j],
                sorted_abs[i]
            )
        ):

            j += 1


        avg_rank = (

            (
                i + 1
            )

            +

            j

        ) / 2.0


        ranks[
            order[
                i:j
            ]
        ] = avg_rank


        i = j


    W_pos = float(

        ranks[
            d > 0
        ].sum()
    )


    W_neg = float(

        ranks[
            d < 0
        ].sum()
    )


    denom = (
        W_pos
        +
        W_neg
    )


    if denom == 0:

        return 0.0


    return (

        W_pos
        -
        W_neg

    ) / denom


def bh_fdr(
    p_values
):

    p = np.asarray(
        p_values,
        dtype=float
    )


    n = len(
        p
    )


    order = np.argsort(
        p
    )


    ranked = p[
        order
    ]


    q = np.empty(
        n,
        dtype=float
    )


    running = 1.0


    for i in range(
        n - 1,
        -1,
        -1
    ):

        rank = (
            i + 1
        )


        value = (

            ranked[
                i
            ]

            *
            n

            /
            rank
        )


        running = min(
            running,
            value
        )


        q[
            order[
                i
            ]
        ] = min(
            running,
            1.0
        )


    return q


def bootstrap_mean_ci(
    values,
    seed,
    n_boot=5000
):

    x = np.asarray(
        values,
        dtype=float
    )


    x = x[
        np.isfinite(
            x
        )
    ]


    if len(
        x
    ) == 0:

        return (
            np.nan,
            np.nan
        )


    rng = np.random.default_rng(
        seed
    )


    means = np.empty(
        n_boot,
        dtype=float
    )


    for b in range(
        n_boot
    ):

        sample = rng.choice(

            x,

            size=len(
                x
            ),

            replace=True
        )


        means[
            b
        ] = np.mean(
            sample
        )


    return (

        float(
            np.quantile(
                means,
                0.025
            )
        ),

        float(
            np.quantile(
                means,
                0.975
            )
        ),
    )


# ============================================================
# 28. PREPARE FIXED Y SPACES
# ============================================================

prepared_y_spaces = {}


for space_name, columns in (
    Y_SPACES.items()
):

    prepared_y_spaces[
        space_name
    ] = prepare_y_space(

        master,

        columns
    )


# ============================================================
# 29. BASE MATRICES
# ============================================================

X_rdkit = (

    master[
        RDKIT_COLUMNS
    ]

    .to_numpy(
        dtype=float
    )
)


X_td_numeric = (

    master[
        TD_NUMERIC_COLUMNS
    ]

    .to_numpy(
        dtype=float
    )
)


td_language_source = (

    master[
        [
            "Canonical_ID",
            *TD_NUMERIC_COLUMNS,
        ]
    ]

    .copy()
)


# ============================================================
# 30. RANDOM SPLITS
# ============================================================

rng = np.random.default_rng(
    RANDOM_STATE
)


splits = []


for repeat in range(
    N_REPEATS
):

    perm = rng.permutation(
        N
    )


    initial_idx = np.sort(
        perm[
            :INITIAL_N
        ]
    )


    holdout_idx = np.sort(
        perm[
            INITIAL_N:
        ]
    )


    if len(
        holdout_idx
    ) != HOLDOUT_N:

        raise RuntimeError(
            "Invalid holdout size."
        )


    splits.append(
        (
            initial_idx,
            holdout_idx
        )
    )


# ============================================================
# 31. RESULT CONTAINERS
# ============================================================

split_rows = []

selection_rows = []

coverage_rows = []

auc_rows = []

diversity_rows = []

td_rule_rows = []

td_sentence_rows = []

block_scale_rows = []


# ============================================================
# 32. MAIN 200-REPEAT LOOP
# ============================================================

print(
    "\nStarting 200-repeat exploration..."
)


for repeat, (
    initial_idx,
    holdout_idx
) in enumerate(
    splits,
    start=1
):


    if (
        repeat == 1

        or

        repeat % 20 == 0
    ):

        print(
            f"Repeat {repeat}/{N_REPEATS}"
        )


    # --------------------------------------------------------
    # Save split
    # --------------------------------------------------------

    for idx in initial_idx:

        split_rows.append(

            {

                "Repeat":
                    repeat,

                "Canonical_ID":
                    ids[
                        idx
                    ],

                "Split":
                    "Initial",
            }
        )


    for idx in holdout_idx:

        split_rows.append(

            {

                "Repeat":
                    repeat,

                "Canonical_ID":
                    ids[
                        idx
                    ],

                "Split":
                    "Holdout",
            }
        )


    # ========================================================
    # 32A. RDKit NUMERIC BLOCK
    # ========================================================

    D_rdkit, scale_rdkit = (
        numeric_block_distance(

            X_rdkit,

            initial_idx
        )
    )


    # ========================================================
    # 32B. TD-NMR NUMERIC BLOCK
    # ========================================================

    D_td_num, scale_td_num = (
        numeric_block_distance(

            X_td_numeric,

            initial_idx
        )
    )


    # ========================================================
    # 32C. CHEMICAL LANGUAGE BLOCK
    # ========================================================

    D_chem_lang, scale_chem = (
        normalize_distance_block(

            D_chem_lang_raw,

            initial_idx
        )
    )


    # ========================================================
    # 32D. TD DYNAMIC-STATE LANGUAGE
    # ========================================================

    td_rules = fit_td_language_rules(

        td_language_source,

        initial_idx
    )


    td_sentences = build_td_language(

        td_language_source,

        td_rules
    )


    # Encode unique text only

    unique_texts = list(
        dict.fromkeys(
            td_sentences
        )
    )


    unique_embeddings = (
        language_model.encode(

            unique_texts,

            normalize_embeddings=True,

            show_progress_bar=False
        )
    )


    text_to_embedding = {

        text:
            emb

        for text, emb in zip(

            unique_texts,

            unique_embeddings
        )
    }


    X_td_lang = np.vstack(

        [

            text_to_embedding[
                text
            ]

            for text in td_sentences
        ]
    )


    D_td_lang_raw = (
        euclidean_distance_matrix(
            X_td_lang
        )
    )


    D_td_lang, scale_td_lang = (
        normalize_distance_block(

            D_td_lang_raw,

            initial_idx
        )
    )


    # --------------------------------------------------------
    # Save training-only rules
    # --------------------------------------------------------

    td_rule_rows.append(

        {

            "Repeat":
                repeat,

            **td_rules,
        }
    )


    # --------------------------------------------------------
    # Save repeat-specific TD sentences
    # --------------------------------------------------------

    for i in range(
        N
    ):

        td_sentence_rows.append(

            {

                "Repeat":
                    repeat,

                "Canonical_ID":
                    ids[
                        i
                    ],

                "Dynamic_State_Language":
                    td_sentences[
                        i
                    ],
            }
        )


    # --------------------------------------------------------
    # Save block scales
    # --------------------------------------------------------

    block_scale_rows.extend(

        [

            {

                "Repeat":
                    repeat,

                "Block":
                    "RDKit_Numeric",

                "MedianTrainingDistance":
                    scale_rdkit,
            },

            {

                "Repeat":
                    repeat,

                "Block":
                    "TD_Numeric",

                "MedianTrainingDistance":
                    scale_td_num,
            },

            {

                "Repeat":
                    repeat,

                "Block":
                    "Chemical_Language",

                "MedianTrainingDistance":
                    scale_chem,
            },

            {

                "Repeat":
                    repeat,

                "Block":
                    "Dynamic_State_Language",

                "MedianTrainingDistance":
                    scale_td_lang,
            },
        ]
    )


    # ========================================================
    # 32E. FOUR REPRESENTATION SPACES
    # ========================================================

    condition_distances = {

        "N0_Numeric":

            combine_distance_blocks(

                [
                    D_rdkit,
                    D_td_num,
                ]
            ),


        "N1_ChemicalLanguage":

            combine_distance_blocks(

                [
                    D_rdkit,
                    D_td_num,
                    D_chem_lang,
                ]
            ),


        "N2_DynamicStateLanguage":

            combine_distance_blocks(

                [
                    D_rdkit,
                    D_td_num,
                    D_td_lang,
                ]
            ),


        "N3_BothLanguages":

            combine_distance_blocks(

                [
                    D_rdkit,
                    D_td_num,
                    D_chem_lang,
                    D_td_lang,
                ]
            ),
    }


    # ========================================================
    # 32F. MAXIMIN FOR EACH CONDITION
    # ========================================================

    for condition in CONDITIONS:

        D_condition = (
            condition_distances[
                condition
            ]
        )


        order = sequential_maximin(

            D_condition,

            initial_idx,

            holdout_idx
        )


        # ----------------------------------------------------
        # Selection order
        # ----------------------------------------------------

        for rank, idx in enumerate(
            order,
            start=1
        ):

            selection_rows.append(

                {

                    "Repeat":
                        repeat,

                    "Condition":
                        condition,

                    "Rank":
                        rank,

                    "Canonical_ID":
                        ids[
                            idx
                        ],
                }
            )


        # ====================================================
        # 32G. Y-SPACE COVERAGE
        # ====================================================

        for space_name in (
            Y_SPACES.keys()
        ):

            (
                _,
                D_y,
                diameter,
            ) = prepared_y_spaces[
                space_name
            ]


            trajectory = coverage_trajectory(

                D_y,

                initial_idx,

                order,

                diameter
            )


            auc = normalized_auc(
                trajectory
            )


            auc_rows.append(

                {

                    "Repeat":
                        repeat,

                    "Condition":
                        condition,

                    "Y_Space":
                        space_name,

                    "Coverage_AUC":
                        auc,
                }
            )


            for k, value in enumerate(
                trajectory
            ):

                coverage_rows.append(

                    {

                        "Repeat":
                            repeat,

                        "Condition":
                            condition,

                        "Y_Space":
                            space_name,

                        "k":
                            k,

                        "Coverage":
                            value,
                    }
                )


        # ====================================================
        # 32H. CHEMICAL-DESIGN DIVERSITY
        # ====================================================

        for k in range(
            1,
            HOLDOUT_N + 1
        ):

            selected_idx = order[
                :k
            ]


            selected_df = master.iloc[
                selected_idx
            ]


            identity_div = (
                role_identity_diversity(
                    selected_df
                )
            )


            ratio_var = (
                composition_ratio_variability(
                    selected_df
                )
            )


            formulation_div = (
                pairwise_formulation_diversity(
                    selected_df
                )
            )


            diversity_rows.append(

                {

                    "Repeat":
                        repeat,

                    "Condition":
                        condition,

                    "k":
                        k,

                    "RoleIdentityDiversity":
                        identity_div,

                    "CompositionRatioVariability":
                        ratio_var,

                    "PairwiseFormulationDiversity":
                        formulation_div,
                }
            )


# ============================================================
# 33. DATAFRAMES
# ============================================================

splits_df = pd.DataFrame(
    split_rows
)

selection_df = pd.DataFrame(
    selection_rows
)

coverage_df = pd.DataFrame(
    coverage_rows
)

auc_df = pd.DataFrame(
    auc_rows
)

diversity_df = pd.DataFrame(
    diversity_rows
)

td_rules_df = pd.DataFrame(
    td_rule_rows
)

td_sentences_df = pd.DataFrame(
    td_sentence_rows
)

block_scales_df = pd.DataFrame(
    block_scale_rows
)


# ============================================================
# 34. AUC SUMMARY
# ============================================================

summary_rows = []


for (
    space_name,
    condition
), sub in (

    auc_df.groupby(
        [
            "Y_Space",
            "Condition",
        ]
    )
):

    values = (
        sub[
            "Coverage_AUC"
        ]
        .to_numpy(
            dtype=float
        )
    )


    ci_low, ci_high = (
        bootstrap_mean_ci(

            values,

            seed=(
                RANDOM_STATE
                +
                len(
                    summary_rows
                )
            )
        )
    )


    summary_rows.append(

        {

            "Y_Space":
                space_name,

            "Condition":
                condition,

            "N":
                len(
                    values
                ),

            "Mean_AUC":
                float(
                    np.mean(
                        values
                    )
                ),

            "Median_AUC":
                float(
                    np.median(
                        values
                    )
                ),

            "SD_AUC":
                float(
                    np.std(
                        values,
                        ddof=1
                    )
                ),

            "CI95_low":
                ci_low,

            "CI95_high":
                ci_high,
        }
    )


auc_summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# 35. PAIRED AUC STATISTICS
# ============================================================

CONTRASTS = [

    (
        "N1_vs_N0",
        "N1_ChemicalLanguage",
        "N0_Numeric",
    ),

    (
        "N2_vs_N0",
        "N2_DynamicStateLanguage",
        "N0_Numeric",
    ),

    (
        "N3_vs_N0",
        "N3_BothLanguages",
        "N0_Numeric",
    ),

    (
        "N3_vs_N1",
        "N3_BothLanguages",
        "N1_ChemicalLanguage",
    ),

    (
        "N3_vs_N2",
        "N3_BothLanguages",
        "N2_DynamicStateLanguage",
    ),
]


stat_rows = []


for space_name in (
    Y_SPACES.keys()
):

    wide = (

        auc_df.loc[
            auc_df[
                "Y_Space"
            ]
            ==
            space_name
        ]

        .pivot(

            index="Repeat",

            columns="Condition",

            values="Coverage_AUC"
        )
    )


    for (
        contrast_name,
        A,
        B,
    ) in CONTRASTS:

        a = wide[
            A
        ].to_numpy(
            dtype=float
        )

        b = wide[
            B
        ].to_numpy(
            dtype=float
        )


        diff = (
            a - b
        )


        try:

            stat, p = wilcoxon(

                diff,

                zero_method="wilcox",

                alternative="two-sided"
            )

        except ValueError:

            stat = 0.0

            p = 1.0


        stat_rows.append(

            {

                "Y_Space":
                    space_name,

                "Contrast":
                    contrast_name,

                "Condition_A":
                    A,

                "Condition_B":
                    B,

                "Mean_A":
                    float(
                        np.mean(
                            a
                        )
                    ),

                "Mean_B":
                    float(
                        np.mean(
                            b
                        )
                    ),

                "Mean_Difference_A_minus_B":
                    float(
                        np.mean(
                            diff
                        )
                    ),

                "Median_Difference":
                    float(
                        np.median(
                            diff
                        )
                    ),

                "Wilcoxon_statistic":
                    float(
                        stat
                    ),

                "p":
                    float(
                        p
                    ),

                "Rank_biserial":
                    float(
                        rank_biserial_paired(
                            diff
                        )
                    ),

                "Win_rate_A_gt_B":
                    float(
                        np.mean(
                            diff > 0
                        )
                    ),

                "Tie_rate":
                    float(
                        np.mean(
                            np.isclose(
                                diff,
                                0.0
                            )
                        )
                    ),
            }
        )


stats_df = pd.DataFrame(
    stat_rows
)


stats_df[
    "q_BH"
] = bh_fdr(
    stats_df[
        "p"
    ].values
)


# ============================================================
# 36. FACTORIAL LANGUAGE EFFECTS
# ============================================================
#
# N0 = neither language
# N1 = Chemical Language
# N2 = Dynamic-State Language
# N3 = both
#
# ============================================================

factorial_rows = []


for space_name in (
    Y_SPACES.keys()
):

    wide = (

        auc_df.loc[
            auc_df[
                "Y_Space"
            ]
            ==
            space_name
        ]

        .pivot(

            index="Repeat",

            columns="Condition",

            values="Coverage_AUC"
        )
    )


    N0 = wide[
        "N0_Numeric"
    ].values

    N1 = wide[
        "N1_ChemicalLanguage"
    ].values

    N2 = wide[
        "N2_DynamicStateLanguage"
    ].values

    N3 = wide[
        "N3_BothLanguages"
    ].values


    effects = {

        "ChemicalLanguage_on_Numeric":
            N1 - N0,

        "DynamicStateLanguage_on_Numeric":
            N2 - N0,

        "ChemicalLanguage_given_TDLanguage":
            N3 - N2,

        "TDLanguage_given_ChemicalLanguage":
            N3 - N1,

        "BothLanguages_vs_Numeric":
            N3 - N0,

        "Interaction":
            N3 - N1 - N2 + N0,
    }


    for effect_name, values in (
        effects.items()
    ):

        factorial_rows.append(

            {

                "Y_Space":
                    space_name,

                "Effect":
                    effect_name,

                "Mean":
                    float(
                        np.mean(
                            values
                        )
                    ),

                "Median":
                    float(
                        np.median(
                            values
                        )
                    ),

                "SD":
                    float(
                        np.std(
                            values,
                            ddof=1
                        )
                    ),

                "Positive_fraction":
                    float(
                        np.mean(
                            values > 0
                        )
                    ),
            }
        )


factorial_df = pd.DataFrame(
    factorial_rows
)


# ============================================================
# 37. EARLY TOP-5 SELECTION PROBABILITY
# ============================================================

early = selection_df.loc[
    selection_df[
        "Rank"
    ]
    <=
    EARLY_K
].copy()


early_counts = (

    early

    .groupby(
        [
            "Condition",
            "Canonical_ID",
        ]
    )

    .size()

    .reset_index(
        name="Top5_Count"
    )
)


early_counts[
    "Top5_Probability"
] = (

    early_counts[
        "Top5_Count"
    ]

    /
    N_REPEATS
)


# ============================================================
# 38. LANDMARK COVERAGE
# ============================================================

landmark_df = (

    coverage_df.loc[
        coverage_df[
            "k"
        ]
        .isin(
            LANDMARK_K
        )
    ]

    .copy()
)


landmark_summary = (

    landmark_df

    .groupby(
        [
            "Y_Space",
            "Condition",
            "k",
        ]
    )[
        "Coverage"
    ]

    .agg(
        [
            "mean",
            "median",
            "std",
        ]
    )

    .reset_index()
)


# ============================================================
# 39. DIVERSITY SUMMARY
# ============================================================

diversity_summary = (

    diversity_df

    .groupby(
        [
            "Condition",
            "k",
        ]
    )[
        [
            "RoleIdentityDiversity",
            "CompositionRatioVariability",
            "PairwiseFormulationDiversity",
        ]
    ]

    .agg(
        [
            "mean",
            "median",
            "std",
        ]
    )
)


diversity_summary.columns = [

    "_".join(
        col
    )

    for col in (
        diversity_summary.columns
    )
]


diversity_summary = (
    diversity_summary
    .reset_index()
)


# ============================================================
# 40. MAIN FIGURE A — COVERAGE AUC
# ============================================================

PRIMARY_SPACE = (
    "Glycerol_x_AlkylChain"
)


primary_auc = (

    auc_df.loc[
        auc_df[
            "Y_Space"
        ]
        ==
        PRIMARY_SPACE
    ]

    .copy()
)


plot_summary = (

    auc_summary_df.loc[
        auc_summary_df[
            "Y_Space"
        ]
        ==
        PRIMARY_SPACE
    ]

    .set_index(
        "Condition"
    )

    .reindex(
        CONDITIONS
    )
)


means = (
    plot_summary[
        "Mean_AUC"
    ]
    .values
)


lower = (

    means

    -

    plot_summary[
        "CI95_low"
    ]
    .values
)


upper = (

    plot_summary[
        "CI95_high"
    ]
    .values

    -

    means
)


fig, ax = plt.subplots(
    figsize=(
        8.5,
        4.8
    )
)


x = np.arange(
    len(
        CONDITIONS
    )
)


ax.bar(

    x,

    means,

    yerr=np.vstack(
        [
            lower,
            upper,
        ]
    ),

    capsize=4
)


ax.set_xticks(
    x
)


ax.set_xticklabels(

    [
        CONDITION_LABELS[
            c
        ]
        for c in CONDITIONS
    ],

    rotation=20,

    ha="right"
)


ax.set_ylabel(
    "Coverage AUC"
)


ax.set_title(

    "Independent Solution-NMR response-space coverage\n"
    "Glycerol × Alkyl-chain"
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


plt.tight_layout()


FIG_A_PNG = (
    OUTPUT_DIR
    /
    "Fig5A_Coverage_AUC.png"
)


FIG_A_PDF = (
    OUTPUT_DIR
    /
    "Fig5A_Coverage_AUC.pdf"
)


plt.savefig(

    FIG_A_PNG,

    dpi=600,

    bbox_inches="tight"
)


plt.savefig(

    FIG_A_PDF,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 41. MAIN FIGURE B — AUC VIOLIN
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        9.0,
        4.8
    )
)


violin_data = [

    primary_auc.loc[
        primary_auc[
            "Condition"
        ]
        ==
        condition,
        "Coverage_AUC"
    ].values

    for condition in CONDITIONS
]


ax.violinplot(

    violin_data,

    positions=np.arange(
        1,
        len(
            CONDITIONS
        )
        +
        1
    ),

    showmeans=False,

    showmedians=True,

    showextrema=False
)


ax.set_xticks(

    np.arange(
        1,
        len(
            CONDITIONS
        )
        +
        1
    )
)


ax.set_xticklabels(

    [
        CONDITION_LABELS[
            c
        ]
        for c in CONDITIONS
    ],

    rotation=20,

    ha="right"
)


ax.set_ylabel(
    "Coverage AUC"
)


ax.set_title(

    "Distribution of coverage performance across 200 repeats"
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


plt.tight_layout()


FIG_B_PNG = (
    OUTPUT_DIR
    /
    "Fig5B_Coverage_AUC_Violin.png"
)


FIG_B_PDF = (
    OUTPUT_DIR
    /
    "Fig5B_Coverage_AUC_Violin.pdf"
)


plt.savefig(

    FIG_B_PNG,

    dpi=600,

    bbox_inches="tight"
)


plt.savefig(

    FIG_B_PDF,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 42. MAIN FIGURE C — COMPOSITION VARIABILITY TOP-5
# ============================================================

top5_div = (

    diversity_df.loc[
        diversity_df[
            "k"
        ]
        ==
        EARLY_K
    ]

    .copy()
)


fig, ax = plt.subplots(
    figsize=(
        9.0,
        4.8
    )
)


violin_data = [

    top5_div.loc[
        top5_div[
            "Condition"
        ]
        ==
        condition,
        "CompositionRatioVariability"
    ].values

    for condition in CONDITIONS
]


ax.violinplot(

    violin_data,

    positions=np.arange(
        1,
        len(
            CONDITIONS
        )
        +
        1
    ),

    showmeans=False,

    showmedians=True,

    showextrema=False
)


ax.set_xticks(

    np.arange(
        1,
        len(
            CONDITIONS
        )
        +
        1
    )
)


ax.set_xticklabels(

    [
        CONDITION_LABELS[
            c
        ]
        for c in CONDITIONS
    ],

    rotation=20,

    ha="right"
)


ax.set_ylabel(

    "Composition-ratio variability\n"
    "(mean SD across three roles)"
)


ax.set_title(

    "Composition variability among the first five selections"
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


plt.tight_layout()


FIG_C_PNG = (
    OUTPUT_DIR
    /
    "Fig5C_Composition_Variability_Top5.png"
)


FIG_C_PDF = (
    OUTPUT_DIR
    /
    "Fig5C_Composition_Variability_Top5.pdf"
)


plt.savefig(

    FIG_C_PNG,

    dpi=600,

    bbox_inches="tight"
)


plt.savefig(

    FIG_C_PDF,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 43. COVERAGE TRAJECTORY FIGURE
# ============================================================

trajectory_summary = (

    coverage_df.loc[
        coverage_df[
            "Y_Space"
        ]
        ==
        PRIMARY_SPACE
    ]

    .groupby(
        [
            "Condition",
            "k",
        ]
    )[
        "Coverage"
    ]

    .agg(
        [
            "mean",
            "std",
        ]
    )

    .reset_index()
)


fig, ax = plt.subplots(
    figsize=(
        7.5,
        5.0
    )
)


for condition in CONDITIONS:

    sub = (

        trajectory_summary.loc[
            trajectory_summary[
                "Condition"
            ]
            ==
            condition
        ]

        .sort_values(
            "k"
        )
    )


    ax.plot(

        sub[
            "k"
        ],

        sub[
            "mean"
        ],

        marker="o",

        markersize=3,

        label=CONDITION_LABELS[
            condition
        ]
    )


ax.set_xlabel(
    "Number of selected holdout materials"
)


ax.set_ylabel(
    "Response-space coverage"
)


ax.set_title(

    "Coverage trajectory: Glycerol × Alkyl-chain"
)


ax.legend(
    frameon=False,
    fontsize=8
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


plt.tight_layout()


TRAJ_PNG = (
    OUTPUT_DIR
    /
    "FigS_Coverage_Trajectory.png"
)


TRAJ_PDF = (
    OUTPUT_DIR
    /
    "FigS_Coverage_Trajectory.pdf"
)


plt.savefig(

    TRAJ_PNG,

    dpi=600,

    bbox_inches="tight"
)


plt.savefig(

    TRAJ_PDF,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 44. PARETO SUMMARY
# ============================================================
#
# Keep response-space coverage and chemical-design diversity
# separate.
#
# Pareto analysis uses:
#
#   mean Coverage AUC
#   mean Top-5 CompositionRatioVariability
#
# No scalar ranking is imposed.
#
# ============================================================

pareto_rows = []


for condition in CONDITIONS:

    mean_auc = float(

        primary_auc.loc[
            primary_auc[
                "Condition"
            ]
            ==
            condition,
            "Coverage_AUC"
        ]
        .mean()
    )


    mean_div = float(

        top5_div.loc[
            top5_div[
                "Condition"
            ]
            ==
            condition,
            "CompositionRatioVariability"
        ]
        .mean()
    )


    pareto_rows.append(

        {

            "Condition":
                condition,

            "Mean_Coverage_AUC":
                mean_auc,

            "Mean_Top5_CompositionVariability":
                mean_div,
        }
    )


pareto_df = pd.DataFrame(
    pareto_rows
)


pareto_df[
    "Pareto_Nondominated"
] = True


for i in range(
    len(
        pareto_df
    )
):

    for j in range(
        len(
            pareto_df
        )
    ):

        if i == j:

            continue


        ai = pareto_df.loc[
            i,
            "Mean_Coverage_AUC"
        ]

        di = pareto_df.loc[
            i,
            "Mean_Top5_CompositionVariability"
        ]


        aj = pareto_df.loc[
            j,
            "Mean_Coverage_AUC"
        ]

        dj = pareto_df.loc[
            j,
            "Mean_Top5_CompositionVariability"
        ]


        dominated = (

            (
                aj >= ai
            )

            and

            (
                dj >= di
            )

            and

            (
                (
                    aj > ai
                )

                or

                (
                    dj > di
                )
            )
        )


        if dominated:

            pareto_df.loc[
                i,
                "Pareto_Nondominated"
            ] = False

            break


# ============================================================
# 45. SAVE CSV OUTPUTS
# ============================================================

csv_outputs = {


    "01_Master_Analysis_Table.csv":
        master,


    "02_Repeated_Splits.csv":
        splits_df,


    "03_Selection_Order.csv":
        selection_df,


    "04_Coverage_Trajectories.csv":
        coverage_df,


    "05_Coverage_AUC.csv":
        auc_df,


    "06_Coverage_AUC_Summary.csv":
        auc_summary_df,


    "07_Paired_Statistics.csv":
        stats_df,


    "08_Factorial_Language_Effects.csv":
        factorial_df,


    "09_Composition_Diversity.csv":
        diversity_df,


    "10_Composition_Diversity_Summary.csv":
        diversity_summary,


    "11_Early_Top5_Selection_Probability.csv":
        early_counts,


    "12_Landmark_Coverage.csv":
        landmark_summary,


    "13_TD_Language_TrainingOnly_Rules.csv":
        td_rules_df,


    "14_TD_DynamicState_Language_AllRepeats.csv":
        td_sentences_df,


    "15_Block_Distance_Normalization.csv":
        block_scales_df,


    "16_Pareto_Summary.csv":
        pareto_df,
}


saved_csv_paths = []


for filename, dataframe in (
    csv_outputs.items()
):

    path = (
        OUTPUT_DIR
        /
        filename
    )


    dataframe.to_csv(

        path,

        index=False,

        encoding="utf-8-sig"
    )


    saved_csv_paths.append(
        path
    )


# ============================================================
# 46. SOURCE DATA EXCEL
# ============================================================

SOURCE_XLSX = (

    OUTPUT_DIR
    /
    "17_Final_Exploration_Source_Data.xlsx"
)


with pd.ExcelWriter(

    SOURCE_XLSX,

    engine="openpyxl"

) as writer:


    auc_summary_df.to_excel(

        writer,

        sheet_name="AUC_summary",

        index=False
    )


    auc_df.to_excel(

        writer,

        sheet_name="AUC_all_repeats",

        index=False
    )


    coverage_df.to_excel(

        writer,

        sheet_name="Coverage_trajectories",

        index=False
    )


    stats_df.to_excel(

        writer,

        sheet_name="Paired_statistics",

        index=False
    )


    factorial_df.to_excel(

        writer,

        sheet_name="Factorial_effects",

        index=False
    )


    top5_div.to_excel(

        writer,

        sheet_name="Top5_composition",

        index=False
    )


    diversity_summary.to_excel(

        writer,

        sheet_name="Diversity_summary",

        index=False
    )


    early_counts.to_excel(

        writer,

        sheet_name="Early_selection",

        index=False
    )


    pareto_df.to_excel(

        writer,

        sheet_name="Pareto",

        index=False
    )


# ============================================================
# 47. METADATA
# ============================================================

metadata = {

    "pipeline":
        "05_Final_exploration_analysis",

    "expected_materials":
        EXPECTED_N,

    "repeats":
        N_REPEATS,

    "initial_n":
        INITIAL_N,

    "holdout_n":
        HOLDOUT_N,

    "random_state":
        RANDOM_STATE,

    "selection":
        "sequential maximin",

    "conditions": {

        "N0":
            "RDKit Numeric + TD-NMR Numeric",

        "N1":
            (
                "RDKit Numeric + TD-NMR Numeric "
                "+ Chemical Language"
            ),

        "N2":
            (
                "RDKit Numeric + TD-NMR Numeric "
                "+ TD Dynamic-State Language"
            ),

        "N3":
            (
                "RDKit Numeric + TD-NMR Numeric "
                "+ Chemical Language "
                "+ TD Dynamic-State Language"
            ),
    },

    "RDKit_dimensions":
        len(
            RDKIT_COLUMNS
        ),

    "TD_numeric_dimensions":
        len(
            TD_NUMERIC_COLUMNS
        ),

    "Chemical_language_dimensions":
        len(
            chem_embedding_cols
        ),

    "Dynamic_State_Language_model":
        LANGUAGE_MODEL_NAME,

    "numeric_scaling":
        (
            "StandardScaler fitted on initial30 "
            "separately for each repeat"
        ),

    "block_normalization":
        (
            "Each block distance matrix divided by "
            "median pairwise distance among initial30"
        ),

    "block_combination":
        (
            "Root mean square of normalized "
            "block distance matrices"
        ),

    "TD_language_thresholds":
        (
            "q33/q67 fitted on initial30 "
            "separately for every repeat"
        ),

    "solution_NMR_used_for_selection":
        False,

    "solution_NMR_role":
        "independent post-selection evaluation",

    "primary_Y_space":
        "Glycerol x AlkylChain",

    "secondary_Y_space":
        "Glycerol x Alkenyl",

    "coverage_metric":
        (
            "1 - RMS nearest-selected Y-space "
            "distance / full-space diameter"
        ),

    "early_selection_k":
        EARLY_K,
}


METADATA_PATH = (

    OUTPUT_DIR
    /
    "18_Metadata.json"
)


with open(

    METADATA_PATH,

    "w",

    encoding="utf-8"

) as f:


    json.dump(

        metadata,

        f,

        indent=2,

        ensure_ascii=False
    )


# ============================================================
# 48. README
# ============================================================

README_PATH = (

    OUTPUT_DIR
    /
    "README.txt"
)


readme = """
05_Final_exploration_analysis

FINAL LEAKAGE-SAFE EXPLORATION PIPELINE

Materials:
43 experimentally characterized copolymers

Repeated design:
200 repeats
30 initial materials
13 holdout materials

Selection:
Sequential maximin

Conditions:
N0 = RDKit Numeric + TD-NMR Numeric
N1 = N0 + Chemical Language
N2 = N0 + Dynamic-State Language
N3 = N0 + both language representations

Leakage control:
Solution-NMR is never used to construct X or select materials.

Numeric transforms:
StandardScaler is fitted using the initial 30 materials only.

Dynamic-State Language:
Thresholds are fitted using the initial 30 materials only
for every repeat.

Distance blocks:
Each block is normalized by the median pairwise distance
within the initial 30 materials.
Normalized block distances are combined by RMS.

Independent evaluation:
Primary:
Glycerol × Alkyl-chain

Secondary:
Glycerol × Alkenyl

Main metrics:
Coverage trajectory
Coverage AUC
Composition-ratio variability
Role-identity diversity
Pairwise formulation diversity
Early Top-5 selection probability

Statistics:
Paired Wilcoxon
BH-FDR
Rank-biserial effect size
Win rate
Factorial language effects
"""


with open(

    README_PATH,

    "w",

    encoding="utf-8"

) as f:

    f.write(
        readme
    )


# ============================================================
# 49. FINAL QC REPORT
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "FINAL ANALYSIS COMPLETED"
)


print(
    "=" * 80
)


print(
    "\nMaterials:",
    N
)


print(
    "Repeats:",
    N_REPEATS
)


print(
    "Initial / holdout:",
    INITIAL_N,
    "/",
    HOLDOUT_N
)


print(
    "\nPrimary Y space:"
)


print(
    PRIMARY_SPACE
)


print(
    "\nMean primary Coverage AUC:"
)


for condition in CONDITIONS:

    value = (

        primary_auc.loc[
            primary_auc[
                "Condition"
            ]
            ==
            condition,
            "Coverage_AUC"
        ]
        .mean()
    )


    print(

        f"{CONDITION_LABELS[condition]:32s} "
        f"{value:.6f}"
    )


print(
    "\nPaired statistics:"
)


print(

    stats_df.loc[
        stats_df[
            "Y_Space"
        ]
        ==
        PRIMARY_SPACE,
        [
            "Contrast",
            "Mean_Difference_A_minus_B",
            "p",
            "q_BH",
            "Rank_biserial",
            "Win_rate_A_gt_B",
        ]
    ]
    .to_string(
        index=False
    )
)


print(
    "\nPareto summary:"
)


print(
    pareto_df.to_string(
        index=False
    )
)


# ============================================================
# 50. ZIP
# ============================================================

all_outputs = (

    saved_csv_paths

    +

    [

        SOURCE_XLSX,

        METADATA_PATH,

        README_PATH,

        FIG_A_PNG,

        FIG_A_PDF,

        FIG_B_PNG,

        FIG_B_PDF,

        FIG_C_PNG,

        FIG_C_PDF,

        TRAJ_PNG,

        TRAJ_PDF,
    ]
)


with zipfile.ZipFile(

    ZIP_PATH,

    "w",

    compression=zipfile.ZIP_DEFLATED

) as z:


    for path in all_outputs:

        z.write(

            path,

            arcname=path.name
        )


print(
    "\nZIP:"
)


print(
    ZIP_PATH
)


# ============================================================
# 51. DOWNLOAD
# ============================================================

try:

    from google.colab import files

    files.download(
        str(
            ZIP_PATH
        )
    )

except ImportError:

    pass

Upload the following files:
1) 01_TD_NMR_descriptors.csv
2) 01_Dynamic_State_Language.csv
3) Final Solution-NMR Y CSV
4) 03_RDKit_Numeric_Final.csv
5) 04_Chemical_Language_Sentences.csv
6) 05_Chemical_Language_Embedding_384D.csv
7) copolymer_composition.csv


Saving 01_Dynamic_State_Language.csv to 01_Dynamic_State_Language.csv
Saving 01_TD_NMR_descriptors.csv to 01_TD_NMR_descriptors.csv
Saving 03_RDKit_Numeric_Final.csv to 03_RDKit_Numeric_Final.csv
Saving 03_SolutionNMR_independent_Y.csv to 03_SolutionNMR_independent_Y.csv
Saving 04_Chemical_Language_Sentences.csv to 04_Chemical_Language_Sentences.csv
Saving 05_Chemical_Language_Embedding_384D.csv to 05_Chemical_Language_Embedding_384D.csv
Saving copolymer_composition.csv to copolymer_composition.csv

Detected inputs
----------------
TD numeric              : 01_TD_NMR_descriptors.csv
TD language audit       : 01_Dynamic_State_Language.csv
Solution-NMR Y          : 03_SolutionNMR_independent_Y.csv
RDKit                   : 03_RDKit_Numeric_Final.csv
Chemical sentences      : 04_Chemical_Language_Sentences.csv
Chemical embedding      : 05_Chemical_Language_Embedding_384D.csv
Composition             : copolymer_composition.csv

Chemical Language dimensions: 384

Merged materials: 43

Loadi

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

Repeat 200/200


/tmp/ipykernel_699/557093183.py:2184: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(
/tmp/ipykernel_699/557093183.py:2184: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(
/tmp/ipykernel_699/557093183.py:2184: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(
/tmp/ipykernel_699/557093183.py:2184: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(



FINAL ANALYSIS COMPLETED

Materials: 43
Repeats: 200
Initial / holdout: 30 / 13

Primary Y space:
Glycerol_x_AlkylChain

Mean primary Coverage AUC:
N0: Numeric                      0.963260
N1: + Chemical Language          0.962411
N2: + Dynamic-State Language     0.963764
N3: + Both Languages             0.962365

Paired statistics:
Contrast  Mean_Difference_A_minus_B            p         q_BH  Rank_biserial  Win_rate_A_gt_B
N1_vs_N0                  -0.000849 1.988972e-07 4.972430e-07      -0.424080            0.345
N2_vs_N0                   0.000504 5.310527e-01 5.310527e-01       0.051864            0.500
N3_vs_N0                  -0.000896 4.594156e-05 6.563079e-05      -0.332338            0.360
N3_vs_N1                  -0.000046 3.943924e-01 4.382138e-01       0.069502            0.510
N3_vs_N2                  -0.001399 1.347797e-08 4.492655e-08      -0.463184            0.340

Pareto summary:
              Condition  Mean_Coverage_AUC  Mean_Top5_CompositionVariability  Pare

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>